# RootPath voices with YarnGPT

This records every line RootPath says, with **Nigerian voices from [YarnGPT](https://github.com/saheedniyi02/yarngpt)**, and packs them into one zip for the app.

**How to run it (about 20–30 minutes for the demo set):**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. If the upload box appears in step 2, upload `voice/manifest.json` from the RootPath repo.
4. At the end, `rootpath-voice.zip` downloads. Send it back to be added to the app.

If Colab disconnects, run all again: clips already made are kept and skipped (while the session lasts).

**If you see an error about `protobuf` or `tensorflow`:** Runtime → **Restart session**, then Runtime → **Run all** again.

Voices: English `idera`, Pidgin `jude` (Nigerian English voice), Yoruba `yoruba_female2`, Igbo `igbo_female2`, Hausa `hausa_female1`. Change them in step 3 if you prefer others.

*Licence note: YarnGPT's Yoruba, Igbo and Hausa voices are licensed for non-commercial use (CC-BY-NC-SA). Fine for the hackathon; ask the author before any commercial launch.*

## 1. Install YarnGPT

In [ ]:
# Must run before anything loads `transformers`: YarnGPT uses PyTorch only, never TensorFlow.
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
!pip -q install outetts uroman gdown inflect
# YarnGPT does not use TensorFlow. outetts installs an old protobuf that crashes Colab's
# TensorFlow when transformers tries to load it, so remove TensorFlow.
!pip -q uninstall -y tensorflow tf-keras 2>/dev/null
!git clone -q https://github.com/saheedniyi02/yarngpt.git 2>/dev/null || echo "yarngpt already cloned"
!wget -q -nc https://huggingface.co/novateur/WavTokenizer-medium-speech-75token/resolve/main/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml
![ -f wavtokenizer_large_speech_320_24k.ckpt ] || gdown -q 1-ASeEkrn4HY49yZWHTASgfGFNXdVnLTt
!ls -lh *.yaml *.ckpt
try:
    from outetts.wav_tokenizer.decoder import WavTokenizer  # needed by YarnGPT
    print("outetts OK")
except Exception as e:
    print("Installing an older outetts that YarnGPT expects:", e)
    !pip -q install "outetts<0.3"
    print("If the next step fails, use Runtime > Restart session, then Run all again.")

## 2. Get the list of lines to record

In [ ]:
import json, urllib.request
# The demo set (what judges hear). For everything, change manifest.json to manifest-full.json.
MANIFEST_URL = "https://raw.githubusercontent.com/Schwoz17/RootPath/main/voice/manifest.json"
try:
    rows = json.load(urllib.request.urlopen(MANIFEST_URL, timeout=30))
    print("Downloaded", len(rows), "lines from GitHub")
except Exception as e:
    print("Could not download (", e, "). Please upload voice/manifest.json from the repo:")
    from google.colab import files
    up = files.upload()
    rows = json.load(open(list(up)[0]))
from collections import Counter
print(Counter(r["lang"] for r in rows))

## 3. Load the model

In [ ]:
import os
os.environ["USE_TF"] = "0"              # make transformers use PyTorch only
os.environ["TRANSFORMERS_NO_TF"] = "1"
import sys, time, subprocess, torch, torchaudio
sys.path.insert(0, "/content")
# Tell transformers TensorFlow is not available, even if it was detected earlier in this session.
import transformers, transformers.utils as _tu, transformers.utils.import_utils as _iu
_iu._tf_available = False
_iu.is_tf_available = lambda: False
_tu.is_tf_available = lambda: False
from transformers import AutoModelForCausalLM
from yarngpt.audiotokenizer import AudioTokenizerV2

MODEL = "saheedniyi/YarnGPT2"
audio_tokenizer = AudioTokenizerV2(MODEL,
    "/content/wavtokenizer_large_speech_320_24k.ckpt",
    "/content/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml")
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto").to(audio_tokenizer.device)
print("Model ready on", audio_tokenizer.device)

# Optional: change a language's voice here, e.g. {"pcm": "emma"} or {"yo": "yoruba_male2"}
SPEAKER_OVERRIDE = {}

def say(text, yarn_lang, speaker):
    prompt = audio_tokenizer.create_prompt(text, lang=yarn_lang, speaker_name=speaker)
    ids = audio_tokenizer.tokenize_prompt(prompt)
    extra = {} if yarn_lang == "english" else {"num_beams": 5}  # beam search helps the local languages
    out = model.generate(input_ids=ids, temperature=0.1, repetition_penalty=1.1, max_length=4000, **extra)
    return audio_tokenizer.get_audio(audio_tokenizer.get_codes(out))

## 4. Listen to a few samples first

In [ ]:
import IPython.display as ipd
for r in [rows[1], next(x for x in rows if x["lang"]=="pcm"), next(x for x in rows if x["lang"]=="yo")]:
    spk = SPEAKER_OVERRIDE.get(r["lang"], r["speaker"])
    print(r["lang"], spk, "|", r["text"])
    ipd.display(ipd.Audio(say(r["text"], r["yarn_lang"], spk).squeeze().cpu().numpy(), rate=24000))

## 5. Record everything (skips lines already done)

In [ ]:
os.makedirs("clips", exist_ok=True)
# To re-record only some clips (e.g. the "bad" list from voice/listen.html), paste their ids here:
REDO_IDS = []          # e.g. ["pcm-b12279c5", "en-1f583592"]
# If you leave REDO_IDS empty, the list of clips to re-record is read from the repo (voice/redo.json).
if not REDO_IDS:
    try:
        REDO_IDS = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/Schwoz17/RootPath/main/voice/redo.json", timeout=30))
    except Exception as e:
        print("No redo list found (", e, ") - recording any missing clips.")
print("Re-recording:", REDO_IDS or "nothing (only missing clips)")
MAX_CHARS_PER_SEC = 19 # faster than this usually means YarnGPT cut the sentence short -> try again
MIN_SECONDS = 0.6      # anything shorter is almost certainly broken -> try again
for rid in REDO_IDS:
    if os.path.exists(f"clips/{rid}.mp3"): os.remove(f"clips/{rid}.mp3")
todo = [r for r in rows if (not REDO_IDS or r["id"] in REDO_IDS) and not os.path.exists(f"clips/{r['id']}.mp3")]
print("To record:", len(todo))

def best_take(r, tries=3):
    """Record up to `tries` times; stop at the first take that is not too fast, else keep the longest."""
    spk = SPEAKER_OVERRIDE.get(r["lang"], r["speaker"]); best = None
    # First recordings start with the steadiest setting. Re-recordings always sample with some
    # randomness: the steady setting gives byte-identical audio every time, so it cannot fix a bad clip.
    temps = [0.4, 0.5, 0.6] if REDO_IDS else [0.1, 0.3, 0.5]
    for t, temp in zip(range(tries), temps):
        prompt = audio_tokenizer.create_prompt(r["text"], lang=r["yarn_lang"], speaker_name=spk)
        ids = audio_tokenizer.tokenize_prompt(prompt)
        extra = {} if r["yarn_lang"] == "english" else {"num_beams": 5}
        out = model.generate(input_ids=ids, temperature=temp, do_sample=bool(REDO_IDS) or t > 0, repetition_penalty=1.1, max_length=4000, **extra)
        audio = audio_tokenizer.get_audio(audio_tokenizer.get_codes(out))
        dur = audio.shape[-1] / 24000
        if best is None or dur > best[1]: best = (audio, dur)
        if dur >= MIN_SECONDS and len(r["text"]) / max(dur, 0.1) <= MAX_CHARS_PER_SEC: break
    return best

def zip_only_redo():
    return bool(REDO_IDS)

t0, done, failed, fast = time.time(), 0, [], []
for i, r in enumerate(todo):
    mp3 = f"clips/{r['id']}.mp3"
    try:
        audio, dur = best_take(r)
        if len(r["text"]) / max(dur, 0.1) > MAX_CHARS_PER_SEC: fast.append((r["id"], r["text"]))
        torchaudio.save("tmp.wav", audio.cpu(), sample_rate=24000)
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", "tmp.wav", "-ac", "1", "-b:a", "40k", mp3], check=True)
        done += 1
    except Exception as e:
        failed.append((r["id"], r["text"], str(e)[:120]))
    if (i + 1) % 10 == 0:
        rate = (time.time() - t0) / max(done, 1)
        print(f"{i+1}/{len(todo)} done · about {rate*(len(todo)-i-1)/60:.0f} min left")
print("Finished. New clips:", done, "· failed:", len(failed), "· still very fast (listen to these):", len(fast))
for f in failed[:10]: print("FAILED", f)
for f in fast[:20]: print("FAST", f)

## 6. Download the zip and send it back

In [ ]:
import glob, zipfile, json
ids = sorted(os.path.basename(p)[:-4] for p in glob.glob("clips/*.mp3"))
if REDO_IDS:  # only send back the clips that were re-recorded
    ids = [i for i in ids if i in REDO_IDS]
json.dump(ids, open("clips/index.json", "w"))
with zipfile.ZipFile("rootpath-voice.zip", "w") as z:
    for i in ids:
        z.write(f"clips/{i}.mp3", f"{i}.mp3")
    z.write("clips/index.json", "index.json")
print(len(ids), "clips,", round(os.path.getsize("rootpath-voice.zip")/1e6, 1), "MB")
from google.colab import files
files.download("rootpath-voice.zip")